# Hyperliquid vault of vaults - waterfall release candidate

Based on: `143-hyperliquid-survivor-first-waterfall.ipynb`, `153-research-hyperliquid-equal-weight-recycle-minimum-hold-cleanup.ipynb`, `154-hyperliquid-waterfall-walk-forward-validation.ipynb`, `155-hyperliquid-waterfall-event-exclusion-stress.ipynb`, `156-hyperliquid-waterfall-execution-friction-stress.ipynb`, `157-hyperliquid-waterfall-universe-perturbation.ipynb`, `trade-executor/strategies/test_only/hyper-ai-test.py`

## Hypothesis

This notebook turns the corrected waterfall branch into a clean release-candidate backtest intended to replace NB153. It keeps the survivor-first `age_ramp` signal, uses plain waterfall sizing, and adds the redemption-aware portfolio target from the live-style `hyper-ai-test.py` strategy so cash handling reflects redeemable capital more realistically.

Full-history indicator loading is intentional in this variant. The original NB158 run was misleading because the backtest loader clipped historical vault data too tightly to the trading window, which caused age-derived indicators to miss the pre-backtest history they depend on.

## Key new insights and what we learnt from this experiment

- The main problem in the earlier NB158 run was not the redemption-aware allocator. It was the data-loading window. The strategy had been given too little pre-backtest history, so the first month incorrectly sat in cash.
- After fixing the loader semantics, NB158 now starts trading immediately. On `2025-08-01` the notebook shows `17` candidate vaults, `17` selected survivors, and `15` open positions.
- Once full history is available, the redemption-aware cash target no longer introduces the large deployment penalty seen in the broken run. Mean accepted deployment recovered to `99.94%` of investable equity and mean cash fell to `2.71%`.

## Summary of results

- The corrected NB158 run finished at `98.48%` cumulative return, `210.25%` CAGR, `3.03` Sharpe, `13.07` Sortino, and `-5.48%` max drawdown.
- This is a major improvement over the broken NB158 run (`60.49%` cumulative return, `118.42%` CAGR, `2.48` Sharpe), confirming that the earlier result was polluted by clipped history.
- NB158 is now clearly ahead of NB153 (`52.06%` cumulative return, `99.81%` CAGR, `2.51` Sharpe, `-4.85%` max drawdown).
- Plain NB143 waterfall still remains stronger on raw backtest performance at about `123.10%` cumulative return, `276.33%` CAGR, `3.51` Sharpe, and `-4.50%` max drawdown, but the gap is now much smaller and no longer explained by a broken start-of-backtest cash bug.

## Robustness of results

- The first-month cash artefact is fixed. Eligibility history now starts years before the backtest, and the universe diagnostics show mature vault inclusion well before August 2025.
- Deployment behaviour is now consistent with the underlying waterfall thesis: `100%` of cycles stayed above `75%` deployment and final-cycle cash was only `3,956.29 USD` on `198,482.51 USD` total equity.
- The remaining trade-off versus NB143 looks real rather than accidental. NB158 still gives up some return and Sharpe in exchange for more live-like redemption-aware cash targeting.
- The corrected conclusion is therefore narrower: the earlier NB158 underperformance was mostly a loader bug, but even after fixing it, the redemption-aware release packaging is still a modest drag on the plain-waterfall optimum rather than a free improvement.

## Notebook setup

- Shared imports, chart output settings, and notebook logging for the clean-up variant.

In [ ]:
import datetime
import logging
import re

import pandas as pd
import plotly.express as px
from IPython.display import display

from tradingstrategy.client import Client
from tradeexecutor.utils.notebook import setup_charting_and_output, OutputMode

client = Client.create_jupyter_client()
setup_charting_and_output(OutputMode.static, image_format='png', width=1500, height=1000)
logger = logging.getLogger('strategy')

: 

## Universe definition

- Use the same Hyperliquid survivor-first vault universe as NB152 so the clean-up stays directly comparable.

In [ ]:
from eth_defi.token import USDC_NATIVE_TOKEN
from tradingstrategy.chain import ChainId
from tradeexecutor.state.identifier import AssetIdentifier

CHAIN_ID = ChainId.hyperliquid

EXCHANGES = ('uniswap-v2', 'uniswap-v3')
SUPPORTING_PAIRS = [
    (ChainId.arbitrum, 'uniswap-v3', 'WETH', 'USDC', 0.0005),
    (ChainId.ethereum, 'uniswap-v3', 'WETH', 'USDC', 0.0005),
    (ChainId.ethereum, 'uniswap-v3', 'WBTC', 'USDC', 0.003),
]
LENDING_RESERVES = None
PREFERRED_STABLECOIN = AssetIdentifier(
    chain_id=ChainId.hyperliquid.value,
    address=USDC_NATIVE_TOKEN[ChainId.hyperliquid].lower(),
    token_symbol='USDC',
    decimals=6,
)
ALLOWED_VAULT_DENOMINATION_TOKENS = {'USDC', 'USDT', 'USDC.e', 'crvUSD', 'USDT0', 'USD₮0', 'USDt', 'USDS'}

print(f'Chain universe: {CHAIN_ID.get_name()}')
print(f'Reserve asset: {PREFERRED_STABLECOIN}')


## Parameters

- Keep the strategy parameters identical to NB152. This notebook changes presentation and diagnostics only.

In [ ]:
from tradeexecutor.strategy.cycle import CycleDuration
from tradeexecutor.strategy.default_routing_options import TradeRouting
from tradeexecutor.strategy.parameters import StrategyParameters
from tradingstrategy.timebucket import TimeBucket


class Parameters:
    #: Notebook identifier for the release-candidate variant replacing NB153.
    id = '158-backtest-hyperliquid-waterfall-release-candidate'

    #: Daily candles match the whole Hyperliquid survivor-first research chain.
    candle_time_bucket = TimeBucket.d1
    #: Daily rebalance cadence is the validated schedule used in NB143 and NB154-NB157.
    cycle_duration = CycleDuration.cycle_1d
    #: HyperEVM is the primary chain for vaults.
    chain_id = CHAIN_ID
    #: HyperEVM is the primary execution chain for the survivor-first release branch.
    primary_chain_id = CHAIN_ID
    #: Keep the same exchange set as NB143, NB153, and the live-style test strategy.
    exchanges = EXCHANGES

    #: NB141-NB157 validated the survivor-first allocator family with a 20-vault basket.
    #: Keep this fixed so NB158 is directly comparable to NB143 and the NB154-NB157 robustness reruns.
    max_assets_in_portfolio = 20
    #: NB143 and NB154-NB157 used 98% target deployment and the corrected waterfall branch held up there.
    #: NB156 showed lower allocation is viable, but not better enough to replace the release default.
    allocation_pct = 0.98
    #: The 12% cap is the survivor-first concentration limit carried through NB143-NB157.
    #: Keep it unchanged because the corrected waterfall case was validated with this exact cap.
    max_concentration_pct = 0.12
    #: NB156 showed tighter per-pool caps reduce both return and deployment.
    #: Keep the 20% pool-cap ceiling from NB143 so we do not reintroduce unnecessary cash drag.
    per_position_cap_of_pool_pct = 0.2
    #: Engine hygiene threshold used across the survivor-first notebooks.
    #: Keep it because the alpha model uses it when cleaning up tiny residual positions.
    min_portfolio_weight_pct = 0.005

    #: Hyperliquid has a hard 5 USD minimum deposit.
    absolute_min_vault_deposit_usd = 5.0
    #: Preserve the old 50 USD buy threshold at 100k initial cash.
    individual_rebalance_min_threshold_of_initial_cash_pct = 0.0005
    #: Preserve the old 10 USD sell threshold at 100k initial cash.
    sell_rebalance_min_threshold_of_initial_cash_pct = 0.0001

    #: NB141 selected this wider survivor-first TVL floor and NB143-NB157 validated the allocator on it.
    #: This is intentionally the survivor-first release setting, not the older NB124/NB126 production threshold.
    min_tvl_usd = 7_500
    #: NB141 also selected this young-vault-inclusive age floor for the survivor-first branch.
    #: Keep it fixed so NB158 stays comparable with the corrected waterfall validation chain.
    min_age = 0.075
    #: The corrected reruns kept `age_ramp` as the surviving signal family and NB154-NB157 validated waterfall on top of it.
    weight_signal = 'age_ramp'
    #: NB143 and the corrected robustness notebooks all used a 0.75-year ramp.
    #: Keep the signal definition unchanged so NB158 isolates the release-candidate packaging, not signal drift.
    age_ramp_period = 0.75

    #: August 2025 remains the canonical mature-universe start used by the release branch.
    backtest_start = datetime.datetime(2025, 8, 1)
    #: Keep the same end date as NB143, NB153, and NB154-NB157 for an apples-to-apples comparison.
    backtest_end = datetime.datetime(2026, 3, 11)
    #: Standard research bankroll for the survivor-first chain.
    initial_cash = 100000
    #: Derived at class creation time from the configured initial cash and the 5 USD hard floor.
    individual_rebalance_min_threshold_usd = max(
        absolute_min_vault_deposit_usd,
        initial_cash * individual_rebalance_min_threshold_of_initial_cash_pct,
    )
    #: Derived at class creation time from the configured initial cash and the 5 USD hard floor.
    sell_rebalance_min_threshold_usd = max(
        absolute_min_vault_deposit_usd,
        initial_cash * sell_rebalance_min_threshold_of_initial_cash_pct,
    )

    #: Default routing is still required by the strategy runtime even though it is not the alpha source.
    routing = TradeRouting.default
    #: Set deliberately high so live and notebook indicator calculations use effectively all available history.
    #: This is an operational correction to the old 120-day default because `age()` and other history-derived indicators
    #: depend on the first available data point and would be silently biased by a truncated lookback window.
    required_history_period = datetime.timedelta(days=365 * 20)
    #: Keep the same live-style slippage assumption as NB153 and `hyper-ai-test.py`.
    slippage_tolerance_pct = 0.0060
    #: Assume no liquidity if there is a gap in TVL data.
    assummed_liquidity_when_data_missings_usd = 0.01


parameters = StrategyParameters.from_class(Parameters)

## Trading universe

- Build the tradable supporting-pair and Hyperliquid vault universe used by the backtest.

In [ ]:
from dotenv import load_dotenv
from tradingstrategy.utils.token_filter import filter_for_selected_pairs
from tradeexecutor.analysis.pair import display_strategy_universe
from tradeexecutor.strategy.execution_context import notebook_execution_context
from tradeexecutor.strategy.pandas_trader.trading_universe_input import CreateTradingUniverseInput
from tradeexecutor.strategy.trading_strategy_universe import TradingStrategyUniverse, load_partial_data, load_vault_universe_with_metadata
from tradeexecutor.strategy.universe_model import UniverseOptions
from tradeexecutor.utils.notebook import display_dataframe_with_html, display_head_and_tail

from tradeexecutor.curator import build_hyperliquid_vault_universe

load_dotenv(override=True)


def create_trading_universe(
    input: CreateTradingUniverseInput,
) -> TradingStrategyUniverse:
    """Create the release-candidate trading universe.

    Keep the backtest trading window fixed to `Parameters.backtest_start` /
    `Parameters.backtest_end`, but let `required_history_period` extend the
    data-loading window backwards so age and other history-derived indicators
    can see the full pre-backtest history for the selected vaults.
    """
    execution_context = input.execution_context
    client = input.client
    timestamp = input.timestamp
    parameters = input.parameters or Parameters
    universe_options = input.universe_options

    debug_printer = logger.info if execution_context.live_trading else print

    chain_id = parameters.primary_chain_id
    debug_printer(f'Preparing trading universe on chain {chain_id.get_name()}')

    all_pairs_df = client.fetch_pair_universe().to_pandas()
    pairs_df = filter_for_selected_pairs(all_pairs_df, SUPPORTING_PAIRS)
    debug_printer(f'We have total {len(all_pairs_df)} pairs in dataset and going to use {len(pairs_df)} pairs for the strategy')

    source_vaults = build_hyperliquid_vault_universe(
        min_tvl=parameters.min_tvl_usd,
        min_age=0.0,
    )
    vault_universe = load_vault_universe_with_metadata(client, vaults=source_vaults)
    vault_universe = vault_universe.limit_to_denomination(ALLOWED_VAULT_DENOMINATION_TOKENS, check_all_vaults_found=True)
    debug_printer(f'Loaded {vault_universe.get_vault_count()} vaults from remote vault metadata, source vaults count: {len(source_vaults)}')

    # `load_partial_data()` now honours `required_history_period` in backtests
    # as a loader-window extension instead of clipping history to the trading window.
    dataset = load_partial_data(
        client=client,
        time_bucket=parameters.candle_time_bucket,
        pairs=pairs_df,
        execution_context=execution_context,
        universe_options=universe_options,
        liquidity=True,
        liquidity_time_bucket=TimeBucket.d1,
        lending_reserves=LENDING_RESERVES,
        vaults=vault_universe,
        vault_history_source="trading-strategy-website",
        check_all_vaults_found=True,
    )

    return TradingStrategyUniverse.create_from_dataset(
        dataset,
        reserve_asset=PREFERRED_STABLECOIN,
        forward_fill=True,
        forward_fill_until=timestamp,
        primary_chain=parameters.primary_chain_id,
    )


universe_input = CreateTradingUniverseInput(
    execution_context=notebook_execution_context,
    client=client,
    timestamp=None,
    parameters=parameters,
    universe_options=UniverseOptions.from_strategy_parameters_class(Parameters, notebook_execution_context),
    execution_model=None,
)

strategy_universe = create_trading_universe(universe_input)
df = display_strategy_universe(
    strategy_universe,
    sort_key='base',
    sort_numeric=False,
    limit=75,
    show_token_risk=True,
    show_tokensniffer=True,
)
display_head_and_tail(df, head_rows=5, tail_rows=5)


## Indicators

- Define the same inclusion and `age_ramp` indicators used by the parent notebook.

In [ ]:
from tradeexecutor.state.identifier import TradingPairIdentifier
from tradeexecutor.state.types import USDollarAmount
from tradeexecutor.strategy.execution_context import ExecutionContext
from tradeexecutor.strategy.pandas_trader.indicator import IndicatorDependencyResolver, IndicatorSource
from tradeexecutor.strategy.pandas_trader.indicator_decorator import IndicatorRegistry
from tradeexecutor.strategy.trading_strategy_universe import TradingStrategyUniverse
from tradeexecutor.analysis.indicator import display_indicators
from tradeexecutor.strategy.pandas_trader.indicator import calculate_and_load_indicators_inline

indicators = IndicatorRegistry()


@indicators.define(source=IndicatorSource.tvl)
def tvl(
    close: pd.Series,
    execution_context: ExecutionContext,
    timestamp: pd.Timestamp,
) -> pd.Series:
    if execution_context.live_trading:
        from tradingstrategy.utils.forward_fill import forward_fill

        df = pd.DataFrame({'close': close})
        df_ff = forward_fill(
            df,
            Parameters.candle_time_bucket.to_frequency(),
            columns=('close',),
            forward_fill_until=timestamp,
        )
        return df_ff['close']

    return close.resample('1h').ffill()


@indicators.define()
def age(close: pd.Series) -> pd.Series:
    inception = close.index[0]
    age_years = (close.index - inception) / pd.Timedelta(days=365.25)
    return pd.Series(age_years, index=close.index)


@indicators.define(dependencies=(tvl,), source=IndicatorSource.dependencies_only_universe)
def tvl_inclusion_criteria(
    min_tvl_usd: USDollarAmount,
    dependency_resolver: IndicatorDependencyResolver,
) -> pd.Series:
    series = dependency_resolver.get_indicator_data_pairs_combined(tvl)
    mask = series >= min_tvl_usd
    mask_true_values_only = mask[mask]
    return mask_true_values_only.groupby(level='timestamp').apply(
        lambda x: x.index.get_level_values('pair_id').tolist()
    )


@indicators.define(dependencies=(age,), source=IndicatorSource.dependencies_only_universe)
def age_inclusion_criteria(
    min_age: float,
    dependency_resolver: IndicatorDependencyResolver,
) -> pd.Series:
    series = dependency_resolver.get_indicator_data_pairs_combined(age)
    mask = series >= min_age
    mask_true_values_only = mask[mask]
    return mask_true_values_only.groupby(level='timestamp').apply(
        lambda x: x.index.get_level_values('pair_id').tolist()
    )


@indicators.define(source=IndicatorSource.strategy_universe)
def trading_availability_criteria(
    strategy_universe: TradingStrategyUniverse,
) -> pd.Series:
    candle_series = strategy_universe.data_universe.candles.df['open']
    return candle_series.groupby(level='timestamp').apply(
        lambda x: x.index.get_level_values('pair_id').tolist()
    )


@indicators.define(
    dependencies=[
        tvl_inclusion_criteria,
        trading_availability_criteria,
        age_inclusion_criteria,
    ],
    source=IndicatorSource.strategy_universe,
)
def inclusion_criteria(
    strategy_universe: TradingStrategyUniverse,
    min_tvl_usd: USDollarAmount,
    min_age: float,
    dependency_resolver: IndicatorDependencyResolver,
) -> pd.Series:
    benchmark_pair_ids = {
        strategy_universe.get_pair_by_human_description(desc).internal_id
        for desc in SUPPORTING_PAIRS
    }

    tvl_series = dependency_resolver.get_indicator_data(
        tvl_inclusion_criteria,
        parameters={'min_tvl_usd': min_tvl_usd},
    )
    trading_availability_series = dependency_resolver.get_indicator_data(trading_availability_criteria)
    age_series = dependency_resolver.get_indicator_data(
        age_inclusion_criteria,
        parameters={'min_age': min_age},
    )

    df = pd.DataFrame(
        {
            'tvl_pair_ids': tvl_series,
            'trading_availability_pair_ids': trading_availability_series,
            'age_pair_ids': age_series,
        }
    )
    df = df.fillna('').apply(list)

    def _combine(row):
        final_set = (
            set(row['tvl_pair_ids'])
            & set(row['trading_availability_pair_ids'])
            & set(row['age_pair_ids'])
        )
        return final_set - benchmark_pair_ids

    union_criteria = df.apply(_combine, axis=1)
    full_index = pd.date_range(
        start=union_criteria.index.min(),
        end=union_criteria.index.max(),
        freq=Parameters.candle_time_bucket.to_frequency(),
    )
    return union_criteria.reindex(full_index, fill_value=[])


@indicators.define(dependencies=(age,), source=IndicatorSource.dependencies_only_per_pair)
def age_ramp_weight(
    pair: TradingPairIdentifier,
    dependency_resolver: IndicatorDependencyResolver,
    age_ramp_period: float = 1.0,
) -> pd.Series:
    vault_age = dependency_resolver.get_indicator_data('age', pair=pair)
    return (vault_age / age_ramp_period).clip(upper=1.0).clip(lower=0.05)


@indicators.define(dependencies=(inclusion_criteria,), source=IndicatorSource.dependencies_only_universe)
def all_criteria_included_pair_count(
    min_tvl_usd: USDollarAmount,
    min_age: float,
    dependency_resolver: IndicatorDependencyResolver,
) -> pd.Series:
    series = dependency_resolver.get_indicator_data(
        "inclusion_criteria",
        parameters={
            "min_tvl_usd": min_tvl_usd,
            "min_age": min_age,
        },
    )
    return series.apply(len)


@indicators.define(dependencies=(tvl_inclusion_criteria,), source=IndicatorSource.dependencies_only_universe)
def tvl_included_pair_count(
    min_tvl_usd: USDollarAmount,
    dependency_resolver: IndicatorDependencyResolver,
) -> pd.Series:
    series = dependency_resolver.get_indicator_data(
        "tvl_inclusion_criteria",
        parameters={
            "min_tvl_usd": min_tvl_usd,
        },
    )
    return series.apply(len)


@indicators.define(dependencies=(age_inclusion_criteria,), source=IndicatorSource.dependencies_only_universe)
def age_included_pair_count(
    min_age: float,
    dependency_resolver: IndicatorDependencyResolver,
) -> pd.Series:
    series = dependency_resolver.get_indicator_data(
        "age_inclusion_criteria",
        parameters={
            "min_age": min_age,
        },
    )
    return series.apply(len)


@indicators.define(source=IndicatorSource.strategy_universe)
def trading_pair_count(
    strategy_universe: TradingStrategyUniverse,
) -> pd.Series:
    benchmark_pair_ids = {
        strategy_universe.get_pair_by_human_description(desc).internal_id
        for desc in SUPPORTING_PAIRS
    }
    series = strategy_universe.data_universe.candles.df["open"]
    swap_index = series.index.swaplevel(0, 1)

    seen_pairs = set()
    seen_data = {}
    for timestamp, pair_id in swap_index:
        if pair_id in benchmark_pair_ids:
            continue
        seen_pairs.add(pair_id)
        seen_data[timestamp] = len(seen_pairs)

    return pd.Series(seen_data.values(), index=list(seen_data.keys()))


display_indicators(indicators)

indicator_data = calculate_and_load_indicators_inline(
    strategy_universe=strategy_universe,
    create_indicators=indicators.create_indicators,
    parameters=parameters,
)


# Trading universe and inclusion criteria

- Review the combined inclusion-criteria chart before moving to the backtest output.
- Follow it with a table of which vaults first entered the investable set.

In [ ]:
from plotly.graph_objects import Figure
from tradeexecutor.strategy.chart.definition import ChartRegistry, ChartKind, ChartInput
from tradeexecutor.strategy.chart.renderer import ChartBacktestRenderingSetup
from tradeexecutor.strategy.chart.standard.trading_universe import available_trading_pairs
from tradeexecutor.strategy.chart.standard.trading_universe import inclusion_criteria_check
from tradeexecutor.strategy.chart.standard.equity_curve import equity_curve as equity_curve_chart
from tradeexecutor.strategy.chart.standard.equity_curve import equity_curve_with_drawdown
from tradeexecutor.strategy.chart.standard.performance_metrics import performance_metrics
from tradeexecutor.strategy.chart.standard.weight import equity_curve_by_asset
from tradeexecutor.strategy.chart.standard.weight import equity_curve_by_chain
from tradeexecutor.strategy.chart.standard.weight import weight_allocation_statistics
from tradeexecutor.strategy.chart.standard.position import positions_at_end
from tradeexecutor.strategy.chart.standard.thinking import last_messages
from tradeexecutor.strategy.chart.standard.alpha_model import alpha_model_diagnostics
from tradeexecutor.strategy.chart.standard.profit_breakdown import trading_pair_breakdown
from tradeexecutor.strategy.chart.standard.trading_metrics import trading_metrics
from tradeexecutor.strategy.chart.standard.interest import vault_statistics
from tradeexecutor.strategy.chart.standard.vault import all_vault_positions

BENCHMARK_PAIRS = SUPPORTING_PAIRS


def equity_curve_with_benchmark(input: ChartInput) -> list[Figure]:
    '''Equity curve with ETH benchmark.'''
    return equity_curve_chart(
        input,
        benchmark_token_symbols=["ETH"],
    )


def inclusion_criteria_check_with_chain(input: ChartInput) -> pd.DataFrame:
    '''Inclusion criteria table with chain shown.'''
    return inclusion_criteria_check(
        input,
        show_chain=True,
    )


def trading_pair_breakdown_with_chain(input: ChartInput) -> pd.DataFrame:
    '''Trading pair breakdown with chain and address.'''
    return trading_pair_breakdown(
        input,
        show_chain=True,
        show_address=True,
    )


def all_vault_positions_by_profit(input: ChartInput) -> pd.DataFrame:
    '''Vault positions sorted by profit.'''
    return all_vault_positions(
        input,
        sort_by="Profit USD",
        sort_ascending=False,
        show_address=True,
    )


charts = ChartRegistry(default_benchmark_pairs=BENCHMARK_PAIRS)
charts.register(available_trading_pairs, ChartKind.indicator_all_pairs)
charts.register(inclusion_criteria_check_with_chain, ChartKind.indicator_all_pairs)
charts.register(equity_curve_with_benchmark, ChartKind.state_all_pairs)
charts.register(equity_curve_with_drawdown, ChartKind.state_all_pairs)
charts.register(performance_metrics, ChartKind.state_all_pairs)
charts.register(equity_curve_by_asset, ChartKind.state_all_pairs)
charts.register(equity_curve_by_chain, ChartKind.state_all_pairs)
charts.register(weight_allocation_statistics, ChartKind.state_all_pairs)
charts.register(positions_at_end, ChartKind.state_all_pairs)
charts.register(last_messages, ChartKind.state_all_pairs)
charts.register(alpha_model_diagnostics, ChartKind.state_all_pairs)
charts.register(trading_pair_breakdown_with_chain, ChartKind.state_all_pairs)
charts.register(trading_metrics, ChartKind.state_all_pairs)
charts.register(vault_statistics, ChartKind.state_all_pairs)
charts.register(all_vault_positions_by_profit, ChartKind.state_all_pairs)

indicator_chart_renderer = ChartBacktestRenderingSetup(
    registry=charts,
    strategy_input_indicators=indicator_data,
    backtest_start_at=Parameters.backtest_start,
    backtest_end_at=Parameters.backtest_end,
)

## Available pairs

- Combined inclusion-criteria chart for the release candidate, matching the NB36 style.
- This single chart shows visible vaults, TVL-qualified vaults, age-qualified vaults, and the final investable count through time.

In [ ]:
fig, df = indicator_chart_renderer.render(available_trading_pairs, with_dataframe=True)
fig.show()
display_head_and_tail(df, head_rows=5, tail_rows=5)


## Included vault details

- Table of when each vault first entered the final investable set, with chain and TVL context.
- This is a detail table for the combined chart above, not a separate inclusion-criteria chart.

In [ ]:
df = indicator_chart_renderer.render(inclusion_criteria_check_with_chain)
display_head_and_tail(df, head_rows=5, tail_rows=5)


# Time range for backtest

- Show how many vaults meet all criteria across the research window.

In [ ]:
series = indicator_data.get_indicator_series(
    "all_criteria_included_pair_count",
)
fig = px.line(
    series,
    title="Vaults meeting all inclusion criteria",
)
fig.update_yaxes(title="Vault count")
fig.update_xaxes(title="Time")
fig.show()

## Backtest

- Run the cleaned-up NB152 variant without changing the strategy logic or parameters.

In [ ]:
from tradeexecutor.backtest.backtest_runner import run_backtest_inline
from tradeexecutor.exchange_account.allocation import (
    calculate_portfolio_target_value,
    get_redeemable_portfolio_capital,
)
from tradeexecutor.state.trade import TradeExecution
from tradeexecutor.statistics.key_metric import (
    calculate_cagr,
    calculate_max_drawdown,
    calculate_sharpe,
    calculate_sortino,
)
from tradeexecutor.strategy.alpha_model import AlphaModel
from tradeexecutor.strategy.execution_context import ExecutionMode
from tradeexecutor.strategy.pandas_trader.strategy_input import StrategyInput
from tradeexecutor.strategy.tvl_size_risk import USDTVLSizeRiskModel
from tradeexecutor.strategy.weighting import weight_passthrouh
from tradeexecutor.utils.dedent import dedent_any
from tradeexecutor.visual.equity_curve import calculate_equity_curve, calculate_returns
from tradeexecutor.curator import is_quarantined


def _extract_optional_int(text: str, label: str) -> int:
    match = re.search(rf"{re.escape(label)}: ([0-9]+)", text)
    return int(match.group(1)) if match else 0


def position_duration_summary(state) -> dict:
    positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
    durations = []
    for position in positions:
        closed_at = position.closed_at if position.closed_at is not None else Parameters.backtest_end
        durations.append((closed_at - position.opened_at).total_seconds() / 86400)
    duration_series = pd.Series(durations, dtype='float64')
    return {
        'position_count': len(positions),
        'positions_lt_3d': int((duration_series < 3).sum()),
        'share_positions_lt_3d': float((duration_series < 3).mean()) if len(duration_series) else float('nan'),
        'median_position_days': float(duration_series.median()) if len(duration_series) else float('nan'),
    }


def extract_selection_diagnostics_df(state) -> pd.DataFrame:
    rows = []
    for unix_ts, messages in sorted(state.visualisation.messages.items(), key=lambda item: item[0]):
        if not messages:
            continue
        message = '\n'.join(messages)
        rows.append(
            {
                'timestamp': datetime.datetime.utcfromtimestamp(unix_ts),
                'open_positions': _extract_optional_int(message, 'Open/about to open positions'),
                'candidate_signals_created': _extract_optional_int(message, 'Candidate signals created'),
                'selected_survivor_signals': _extract_optional_int(message, 'Selected survivor signals'),
            }
        )
    return pd.DataFrame(rows).set_index('timestamp')


def decide_trades(input: StrategyInput) -> list[TradeExecution]:
    """Run survivor-first capped waterfall sizing with a redemption-aware target value."""
    parameters = input.parameters
    position_manager = input.get_position_manager()
    state = input.state
    timestamp = input.timestamp
    indicators = input.indicators
    strategy_universe = input.strategy_universe

    portfolio = position_manager.get_current_portfolio()
    equity = portfolio.get_total_equity()

    if input.execution_context.mode == ExecutionMode.backtesting and equity < parameters.initial_cash * 0.10:
        return []

    alpha_model = AlphaModel(
        timestamp,
        close_position_weight_epsilon=parameters.min_portfolio_weight_pct,
    )

    tvl_included_pair_count = indicators.get_indicator_value('tvl_included_pair_count')
    included_pairs = indicators.get_indicator_value('inclusion_criteria', na_conversion=False)
    if included_pairs is None:
        included_pairs = []

    signal_count = 0
    for pair_id in included_pairs:
        pair = strategy_universe.get_pair_by_id(pair_id)
        if not state.is_good_pair(pair):
            continue
        if is_quarantined(pair.pool_address, timestamp):
            continue
        age_ramp_weight = indicators.get_indicator_value('age_ramp_weight', pair=pair)
        weight_signal_value = age_ramp_weight if age_ramp_weight is not None else 1.0
        alpha_model.set_signal(pair, weight_signal_value)
        signal_count += 1

    redeemable_capital = get_redeemable_portfolio_capital(position_manager)
    portfolio_target_value = calculate_portfolio_target_value(
        position_manager,
        parameters.allocation_pct,
    )

    alpha_model.select_top_signals(count=parameters.max_assets_in_portfolio)
    alpha_model.assign_weights(method=weight_passthrouh)

    # Hyperliquid vaults are expected to have complete TVL data in live trading.
    # Fail the cycle loudly if TVL is missing instead of silently sizing with a placeholder.
    size_risk_model = USDTVLSizeRiskModel(
        pricing_model=input.pricing_model,
        per_position_cap=parameters.per_position_cap_of_pool_pct,
    )

    alpha_model.normalise_weights(
        investable_equity=portfolio_target_value,
        size_risk_model=size_risk_model,
        max_weight=parameters.max_concentration_pct,
        max_positions=parameters.max_assets_in_portfolio,
        waterfall=True,
    )
    alpha_model.update_old_weights(state.portfolio, ignore_credit=False)
    alpha_model.calculate_target_positions(position_manager)

    trades = alpha_model.generate_rebalance_trades_and_triggers(
        position_manager,
        min_trade_threshold=parameters.individual_rebalance_min_threshold_usd,
        individual_rebalance_min_threshold=parameters.individual_rebalance_min_threshold_usd,
        sell_rebalance_min_threshold=parameters.sell_rebalance_min_threshold_usd,
        execution_context=input.execution_context,
    )

    if input.is_visualisation_enabled():
        try:
            top_signal = next(iter(alpha_model.get_signals_sorted_by_weight()))
            if top_signal.normalised_weight == 0:
                top_signal = None
        except StopIteration:
            top_signal = None

        rebalance_volume = sum(trade.get_value() for trade in trades)
        report = dedent_any(
            f"""
            Cycle: #{input.cycle}
            Rebalanced: {'👍' if alpha_model.is_rebalance_triggered() else '👎'}
            Open/about to open positions: {len(state.portfolio.open_positions)}
            Max position value change: {alpha_model.max_position_adjust_usd:,.2f} USD
            Rebalance threshold: {alpha_model.position_adjust_threshold_usd:,.2f} USD
            Trades decided: {len(trades)}
            Pairs meeting inclusion criteria: {len(included_pairs)}
            Pairs meeting TVL inclusion criteria: {tvl_included_pair_count}
            Candidate signals created: {signal_count}
            Selected survivor signals: {len(alpha_model.signals)}
            Weight signal: {parameters.weight_signal}
            Age ramp period: {parameters.age_ramp_period}
            Total equity: {portfolio.get_total_equity():,.2f} USD
            Cash: {position_manager.get_current_cash():,.2f} USD
            Redeemable capital: {redeemable_capital:,.2f} USD
            Pending redemptions: {position_manager.get_pending_redemptions():,.2f} USD
            Investable equity: {alpha_model.investable_equity:,.2f} USD
            Accepted investable equity: {alpha_model.accepted_investable_equity:,.2f} USD
            Allocated to signals: {alpha_model.get_allocated_value():,.2f} USD
            Discarded allocation because of lack of lit liquidity: {alpha_model.size_risk_discarded_value:,.2f} USD
            Rebalance volume: {rebalance_volume:,.2f} USD
            """
        )

        if top_signal:
            assert top_signal.position_size_risk
            report += dedent_any(
                f"""
                Top signal pair: {top_signal.pair.get_ticker()}
                Top signal value: {top_signal.signal}
                Top signal weight: {top_signal.raw_weight}
                Top signal weight (normalised): {top_signal.normalised_weight * 100:.2f} % (got {top_signal.position_size_risk.get_relative_capped_amount() * 100:.2f} % of asked size)
                """
            )

        for flag, count in alpha_model.get_flag_diagnostics_data().items():
            report += f'Signals with flag {flag.name}: {count}' + '\n'

        state.visualisation.add_message(timestamp, report)
        state.visualisation.set_discardable_data('alpha_model', alpha_model)

    return trades


result = run_backtest_inline(
    name=Parameters.id,
    engine_version='0.5',
    decide_trades=decide_trades,
    create_indicators=indicators.create_indicators,
    cycle_duration=CycleDuration.from_timebucket(parameters.candle_time_bucket),
    client=client,
    universe=strategy_universe,
    parameters=parameters,
    max_workers=1,
    start_at=Parameters.backtest_start,
    end_at=Parameters.backtest_end,
)

state = result.state
selection_diagnostics_df = extract_selection_diagnostics_df(state)
duration_summary = position_duration_summary(state)

equity_curve = calculate_equity_curve(state)
returns = calculate_returns(equity_curve)
summary_df = pd.DataFrame(
    [
        {
            'Cumulative return': equity_curve.iloc[-1] / equity_curve.iloc[0] - 1,
            'CAGR': calculate_cagr(returns),
            'Sharpe': calculate_sharpe(returns, periods=365),
            'Sortino': calculate_sortino(returns, periods=365),
            'Max DD': calculate_max_drawdown(returns),
            'Trades': len(list(state.portfolio.get_all_trades())),
            'Positions': duration_summary['position_count'],
            'Final candidate signals': selection_diagnostics_df['candidate_signals_created'].iloc[-1],
            'Final selected survivors': selection_diagnostics_df['selected_survivor_signals'].iloc[-1],
            'Mean candidate signals': selection_diagnostics_df['candidate_signals_created'].mean(),
            'Mean selected survivors': selection_diagnostics_df['selected_survivor_signals'].mean(),
            'Positions < 3d': duration_summary['positions_lt_3d'],
            'Share positions < 3d': duration_summary['share_positions_lt_3d'],
            'Median position days': duration_summary['median_position_days'],
        }
    ]
)
display(summary_df)


## Latest cycle cash and redemption snapshot

In [ ]:
import re
import pandas as pd


def _extract_optional_usd(text: str, label: str) -> float:
    match = re.search(rf"{re.escape(label)}: ([0-9,]+(?:\.[0-9]+)?) USD", text)
    return float(match.group(1).replace(',', '')) if match else float('nan')


message_map = state.visualisation.get_messages_tail(2)
latest_cycle_label = list(message_map.keys())[0]
latest_message = list(message_map.values())[0]

if 'chart_renderer' not in globals():
    chart_renderer = ChartBacktestRenderingSetup(
        registry=charts,
        strategy_input_indicators=indicator_data,
        state=state,
        backtest_start_at=Parameters.backtest_start,
        backtest_end_at=Parameters.backtest_end,
    )

alpha_df = chart_renderer.render(alpha_model_diagnostics)
if isinstance(alpha_df.columns, pd.MultiIndex):
    alpha_df.columns = [c[0] for c in alpha_df.columns]

accepted_size_col = next(c for c in alpha_df.columns if 'Accepted size' in str(c))
flags_col = next(c for c in alpha_df.columns if 'Flags' in str(c))

accepted_sizes = pd.to_numeric(alpha_df[accepted_size_col], errors='coerce').fillna(0.0)
accepted_mask = accepted_sizes > 0
flags = alpha_df[flags_col].fillna('').astype(str)

total_equity = _extract_optional_usd(latest_message, 'Total equity')
cash = _extract_optional_usd(latest_message, 'Cash')
redeemable_capital = _extract_optional_usd(latest_message, 'Redeemable capital')
pending_redemptions = _extract_optional_usd(latest_message, 'Pending redemptions')
investable = _extract_optional_usd(latest_message, 'Investable equity')
accepted_investable = _extract_optional_usd(latest_message, 'Accepted investable equity')
allocated_to_signals = _extract_optional_usd(latest_message, 'Allocated to signals')
discarded_lit_liquidity = _extract_optional_usd(latest_message, 'Discarded allocation because of lack of lit liquidity')

benchmark_df = pd.DataFrame(
    [
        ('Final cycle', latest_cycle_label),
        ('Total equity USD', total_equity),
        ('Cash USD', cash),
        ('Cash % of equity', cash / total_equity if total_equity else float('nan')),
        ('Redeemable capital USD', redeemable_capital),
        ('Pending redemptions USD', pending_redemptions),
        ('Investable equity USD', investable),
        ('Accepted investable equity USD', accepted_investable),
        ('Accepted / investable %', accepted_investable / investable if investable else float('nan')),
        ('Allocated to signals USD', allocated_to_signals),
        ('Discarded because of lit liquidity USD', discarded_lit_liquidity),
        ('Unaccepted investable USD', investable - accepted_investable),
        ('Unaccepted / investable %', (investable - accepted_investable) / investable if investable else float('nan')),
        ('Accepted positions', int(accepted_mask.sum())),
        ('Mean accepted position USD', accepted_sizes[accepted_mask].mean()),
        ('Median accepted position USD', accepted_sizes[accepted_mask].median()),
        ('Flags: capped_by_pool_size', int(flags.str.contains('capped_by_pool_size').sum())),
        ('Flags: individual_trade_size_too_small', int(flags.str.contains('individual_trade_size_too_small').sum())),
    ],
    columns=['Metric', 'Value'],
)

display(benchmark_df)


## Capital utilisation through time

In [ ]:
import datetime
import re
import pandas as pd


def _extract_optional_usd_series(text: str, label: str) -> float:
    match = re.search(rf"{re.escape(label)}: ([0-9,]+(?:\.[0-9]+)?) USD", text)
    return float(match.group(1).replace(',', '')) if match else float('nan')


def _extract_optional_int_series(text: str, label: str) -> int:
    match = re.search(rf"{re.escape(label)}: ([0-9]+)", text)
    return int(match.group(1)) if match else 0


rows = []
message_tuples = sorted(state.visualisation.messages.items(), key=lambda x: x[0])
for unix_ts, messages in message_tuples:
    if not messages:
        continue
    message = '\n'.join(messages)
    dt = datetime.datetime.utcfromtimestamp(unix_ts)
    total_equity = _extract_optional_usd_series(message, 'Total equity')
    cash = _extract_optional_usd_series(message, 'Cash')
    redeemable = _extract_optional_usd_series(message, 'Redeemable capital')
    pending = _extract_optional_usd_series(message, 'Pending redemptions')
    investable = _extract_optional_usd_series(message, 'Investable equity')
    accepted = _extract_optional_usd_series(message, 'Accepted investable equity')
    allocated = _extract_optional_usd_series(message, 'Allocated to signals')
    discarded = _extract_optional_usd_series(message, 'Discarded allocation because of lack of lit liquidity')
    rows.append({
        'timestamp': dt,
        'total_equity_usd': total_equity,
        'cash_usd': cash,
        'cash_pct_of_equity': cash / total_equity if total_equity else float('nan'),
        'redeemable_capital_usd': redeemable,
        'pending_redemptions_usd': pending,
        'investable_equity_usd': investable,
        'accepted_investable_equity_usd': accepted,
        'accepted_pct_of_investable': accepted / investable if investable else float('nan'),
        'unaccepted_pct_of_investable': (investable - accepted) / investable if investable else float('nan'),
        'allocated_to_signals_usd': allocated,
        'discarded_due_to_lit_liquidity_usd': discarded,
        'capped_by_pool_size_count': _extract_optional_int_series(message, 'Signals with flag capped_by_pool_size'),
        'individual_trade_size_too_small_count': _extract_optional_int_series(message, 'Signals with flag individual_trade_size_too_small'),
    })

utilisation_df = pd.DataFrame(rows).set_index('timestamp')
display(utilisation_df.tail(10))


## Capital utilisation summary

In [ ]:
utilisation_summary = pd.DataFrame(
    [
        ('Mean accepted / investable %', utilisation_df['accepted_pct_of_investable'].mean()),
        ('Median accepted / investable %', utilisation_df['accepted_pct_of_investable'].median()),
        ('Min accepted / investable %', utilisation_df['accepted_pct_of_investable'].min()),
        ('Max accepted / investable %', utilisation_df['accepted_pct_of_investable'].max()),
        ('Mean cash % of equity', utilisation_df['cash_pct_of_equity'].mean()),
        ('Median cash % of equity', utilisation_df['cash_pct_of_equity'].median()),
        ('Max cash % of equity', utilisation_df['cash_pct_of_equity'].max()),
        ('Mean redeemable capital USD', utilisation_df['redeemable_capital_usd'].mean()),
        ('Mean pending redemptions USD', utilisation_df['pending_redemptions_usd'].mean()),
        ('Mean capped_by_pool_size count', utilisation_df['capped_by_pool_size_count'].mean()),
        ('Mean too-small count', utilisation_df['individual_trade_size_too_small_count'].mean()),
        ('Share of cycles above 50% deployment', (utilisation_df['accepted_pct_of_investable'] > 0.50).mean()),
        ('Share of cycles above 75% deployment', (utilisation_df['accepted_pct_of_investable'] > 0.75).mean()),
    ],
    columns=['Metric', 'Value'],
)
display(utilisation_summary)


## Capital utilisation charts

In [ ]:
import plotly.express as px

plot_df = utilisation_df.reset_index()

fig = px.line(
    plot_df,
    x='timestamp',
    y=['accepted_pct_of_investable', 'unaccepted_pct_of_investable', 'cash_pct_of_equity'],
    title='Capital utilisation and cash share over time',
)
fig.update_yaxes(tickformat='.0%')
fig.show()

fig = px.line(
    plot_df,
    x='timestamp',
    y=['cash_usd', 'redeemable_capital_usd', 'pending_redemptions_usd'],
    title='Cash, redeemable capital, and pending redemptions over time',
)
fig.show()

fig = px.line(
    plot_df,
    x='timestamp',
    y=['capped_by_pool_size_count', 'individual_trade_size_too_small_count'],
    title='Which bottleneck dominates over time',
)
fig.show()


## Survivor selection

In [ ]:
display_head_and_tail(selection_diagnostics_df, head_rows=5, tail_rows=5)

plot_df = selection_diagnostics_df.reset_index().rename(
    columns={
        'candidate_signals_created': 'Candidate signals',
        'selected_survivor_signals': 'Selected survivors',
        'open_positions': 'Open positions',
    }
)
fig = px.line(
    plot_df,
    x='timestamp',
    y=['Candidate signals', 'Selected survivors', 'Open positions'],
    title='Survivor selection through time',
)
fig.update_yaxes(title='Vault count')
fig.update_xaxes(title='Time')
fig.show()


# Performance summary

- Start with the notebook-specific summary table, then inspect the standard strategy performance view.

In [ ]:
chart_renderer = ChartBacktestRenderingSetup(
    registry=charts,
    strategy_input_indicators=indicator_data,
    state=state,
    backtest_start_at=Parameters.backtest_start,
    backtest_end_at=Parameters.backtest_end,
)

display(summary_df)

df = chart_renderer.render(performance_metrics)
display(df)


# Equity curve

- Compare the strategy equity curve against the ETH benchmark.

In [ ]:
fig = chart_renderer.render(equity_curve_with_benchmark)
fig.show()

## Equity curve with drawdown

- View the same equity path with drawdown overlaid.

In [ ]:
fig = chart_renderer.render(equity_curve_with_drawdown)
display(fig)

# Asset weights

- Review how the portfolio distributes risk across vaults and chains.

## Portfolio equity curve breakdown by asset

- Show how the main vault holdings contribute to the cumulative equity curve.

In [ ]:
fig = chart_renderer.render(equity_curve_by_asset)
fig.show()

## Portfolio equity curve breakdown by chain

- Break the realised equity curve down by chain exposure.

In [ ]:
fig, chain_breakdown_df = chart_renderer.render(equity_curve_by_chain)
fig.show()



## Weight allocation statistics

- Keep the compact allocation statistics table as a quick summary.

In [ ]:
from tradeexecutor.utils.notebook import display_dataframe_with_html

stats = chart_renderer.render(weight_allocation_statistics)
display_dataframe_with_html(stats)

# Rolling performance

- Track the medium-term return profile and risk-adjusted return profile through time using a fixed 3-month window.


In [ ]:
from tradeexecutor.visual.equity_curve import (
    calculate_rolling_returns,
    calculate_rolling_sharpe,
)

rolling_returns = calculate_rolling_returns(
    returns,
    freq="D",
    periods=90,
)

fig = px.line(rolling_returns, title="Strategy rolling compounded return (3 months)")
fig.update_layout(showlegend=False)
fig.update_yaxes(title="Compounded return", tickformat=".0%")
fig.update_xaxes(title="Time")
fig.show()

rolling_sharpe = calculate_rolling_sharpe(
    returns,
    freq="D",
    periods=90,
)

fig = px.line(rolling_sharpe, title="Strategy rolling Sharpe (3 months)")
fig.update_layout(showlegend=False)
fig.update_yaxes(title="Sharpe")
fig.update_xaxes(title="Time")
fig.show()


# Monthly returns

- Check whether the path of returns is broad-based across months or dominated by a few bursts.


In [ ]:
from tradeexecutor.visual.equity_curve import visualise_returns_over_time

monthly_returns_fig = visualise_returns_over_time(returns)
display(monthly_returns_fig)


# Positions at the end

- Inspect the live basket at the end of the backtest window.

In [ ]:
stats = chart_renderer.render(positions_at_end)
display_head_and_tail(stats, head_rows=10, tail_rows=10)


# Strategy thinking

- Sample the backtest cycle messages as head and tail slices instead of dumping the full table.

In [ ]:
df = chart_renderer.render(last_messages)
display_head_and_tail(df, head_rows=5, tail_rows=5)


# Alpha model diagnostics

- Inspect the alpha-model diagnostics table in compact form.

In [ ]:
df = chart_renderer.render(alpha_model_diagnostics)
display_head_and_tail(df, head_rows=5, tail_rows=5)


# Trading pair breakdown

- Review the vault-level realised contribution table with chain and address context.

In [ ]:
df = chart_renderer.render(trading_pair_breakdown_with_chain)
display_head_and_tail(df, head_rows=10, tail_rows=10)


# Trading metrics

- Keep the full trading metrics summary readable by slicing long output.

In [ ]:
df = chart_renderer.render(trading_metrics)
display_head_and_tail(df, head_rows=10, tail_rows=10)


# Vault performance

- Inspect the vault-interest and position-level views for the executed backtest.

## Vault statistics

- Keep the compact vault interest summary as a full table.

In [ ]:
df = chart_renderer.render(vault_statistics)
display_head_and_tail(df, head_rows=5, tail_rows=5)


## Vault position list

- Show the most profitable and least profitable vault-position rows without dumping the full table.

In [ ]:
df = chart_renderer.render(all_vault_positions_by_profit)
display_head_and_tail(df, head_rows=10, tail_rows=10)
